# Suite CNF — Conflicting rules

A grid's rules are its children's adjacency specs, in the order they reached it. After each change it settles: each rule kept where it holds, met where one change of the sequence breaks no earlier rule that holds, and ignored, the later of the incompatible rules, where none can, until a change lets it be met, in its turn: mbse-basis [CAP-240: conflicting layout rules ignored and requeued](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-240), for container layout, by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import json

from mbse.Schemas.Framework import Builders, Validators
from mbse.Diagrams.Framework import Container, Logical

store = Builders.Store()
store.transaction(Logical.install)
store.transaction(Container.install)
L = store.Logical
diagrams = store.singleton("Logical.Diagrams")
problems = Validators.Validate(store).Reachable

with store.transaction():
    L.DiagramDirectory(diagrams, lambda f: f.diagrams(lambda a: a.insert(lambda e: e.name("overview").diagram(lambda d: d))))
overview = diagrams.diagrams({"name": "overview"})


def box(name, children, span, grows="rows", cell_rotation=0.0):
    """A top-level shape of the diagram holding a child of each name in `children`, laid out in a grid."""
    def spec(s):
        for child in children:
            s.children(lambda k, child=child: k.insert(lambda e: e.name(child).child(lambda c: c)))
        return s
    with store.transaction():
        L.OfDiagram(overview, lambda d: d.shapes(lambda s: s.insert(lambda e: e.name(name).shape(spec))))
        shape = overview.shapes({"name": name})
        Container.layout(shape, span, grows, cell_rotation)
    return shape


def beside(shape, a, b, degrees):
    """Adds the adjacency spec from child `a` to child `b`, by `degrees`, and adjoins them: whether it did."""
    with store.transaction():
        L.OfShape(shape.children({"name": a}), lambda s: s.adjacent_to(lambda x: x.insert(
            lambda e: e.to_shape(shape.children({"name": b})).rotation(lambda r: r.degrees(degrees)))))
        return Container.adjoined(shape.children({"name": a}), shape.children({"name": b}))


def apart(shape, a, b):
    """Removes the adjacency spec from child `a` to child `b`, and unjoins `b`."""
    with store.transaction():
        L.OfShape(shape.children({"name": a}), lambda s: s.adjacent_to(lambda x: x.remove(
            lambda e: e.to_shape(shape.children({"name": b})))))
        Container.unjoined(shape.children({"name": b}))


def cells(shape):
    """Each cell, in row and column order, as JSON text: its row and column, the names of the children it holds, in
    order, each with its order, and its axis and extents."""
    (entry,) = shape.grid
    names = {id(e.child): e.name.get() for e in shape.children}
    found = [[[e.row.get(), e.column.get()], [[names[id(h.shape)], h.order.get()] for h in e.cell.children],
              e.cell.grows.get(), e.cell.rows.get(), e.cell.columns.get()]
             for e in sorted(entry.grid.cells, key=lambda e: (e.row.get(), e.column.get()))]
    return json.dumps(found, separators=(",", ":"))

def queue(shape):
    """The names of the ignored rules' shapes, and their degrees, in the order the rules reached the grid."""
    names = {id(e.child): e.name.get() for e in shape.children}
    return [[names[id(f)], names[id(t)], d] for f, t, d in Container.ignored(shape)]


def rules(shape):
    """The grid's rules, as data: each one's shapes' names, its degrees and whether it is ignored."""
    (entry,) = shape.grid
    names = {id(e.child): e.name.get() for e in shape.children}
    return [[names[id(e.from_shape)], names[id(e.to_shape)], e.degrees.get(), e.ignored.get()] for e in entry.grid.rules]


def grow(shape, name):
    """Adds a child of `name` to `shape`, and places it."""
    with store.transaction():
        L.OfShape(shape, lambda s: s.children(lambda k: k.insert(lambda e: e.name(name).child(lambda c: c))))
        Container.added(shape.children({"name": name}))


<a id="CNF-01"></a>
## CNF-01 · A spec that would break an earlier rule that holds is ignored, and the earlier one kept

In [ ]:
cpu = box("cpu", "abcdef", 3)
assert json.dumps([beside(cpu, "a", "d", 180.0), beside(cpu, "a", "b", 270.0), beside(cpu, "a", "c", 270.0)], separators=(',', ':')) == '[true,false,false]'
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1],[[1,2],[["f",5]],null,1,1]]'
assert json.dumps(queue(cpu), separators=(',', ':')) == '[["a","b",270.0],["a","c",270.0]]'

<a id="CNF-02"></a>
## CNF-02 · A grid's rules are data: each spec, in the order it reached the grid, and whether it is ignored

In [ ]:
assert json.dumps(rules(cpu), separators=(',', ':')) == '[["a","d",180.0,false],["a","b",270.0,true],["a","c",270.0,true]]'

<a id="CNF-03"></a>
## CNF-03 · Removing the earlier spec lifts the conflict: the ignored rules are met, in the order they reached the grid

In [ ]:
apart(cpu, "a", "d")
assert cells(cpu) == '[[[0,0],[["b",1],["c",2]],"rows",2,1],[[0,1],[["a",0]],null,1,1],[[0,2],[["d",3]],null,1,1],[[1,0],[["e",4]],null,1,1],[[1,1],[["f",5]],null,1,1]]'
assert json.dumps(queue(cpu), separators=(',', ':')) == '[]'
assert json.dumps(rules(cpu), separators=(',', ':')) == '[["a","b",270.0,false],["a","c",270.0,false]]'

<a id="CNF-04"></a>
## CNF-04 · Removing a child lifts a conflict its rules made

In [ ]:
gpu = box("gpu", "abcd", 3)
assert json.dumps([beside(gpu, "a", "d", 180.0), beside(gpu, "a", "b", 270.0), queue(gpu)], separators=(',', ':')) == '[true,false,[["a","b",270.0]]]'
with store.transaction():
    L.OfShape(gpu, lambda s: s.children(lambda k: k.remove(lambda e: e.name("d"))))
    Container.removed(gpu)
assert cells(gpu) == '[[[0,0],[["b",1]],null,1,1],[[0,1],[["a",0]],null,1,1],[[0,2],[["c",2]],null,1,1]]'
assert json.dumps(queue(gpu), separators=(',', ':')) == '[]'

<a id="CNF-05"></a>
## CNF-05 · Adding children lifts a conflict with the grid's extents

In [ ]:
mmu = box("mmu", "ab", 3)
assert json.dumps([beside(mmu, "a", "b", 180.0), queue(mmu)], separators=(',', ':')) == '[false,[["a","b",180.0]]]'
grow(mmu, "c")
assert cells(mmu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1]]'
assert json.dumps(queue(mmu), separators=(',', ':')) == '[["a","b",180.0]]'
grow(mmu, "d")
assert cells(mmu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["d",3]],null,1,1],[[1,0],[["b",1]],null,1,1]]'
assert json.dumps(queue(mmu), separators=(',', ':')) == '[]'

<a id="CNF-06"></a>
## CNF-06 · A child reverting out of a shared cell leaves the rules that hold in place

In [ ]:
io = box("io", "pqrst", 3)
assert json.dumps([beside(io, "p", "r", 180.0), beside(io, "p", "t", 180.0)], separators=(',', ':')) == '[true,true]'
assert cells(io) == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[0,2],[["s",3]],null,1,1],[[1,0],[["r",2],["t",4]],"rows",2,1]]'
apart(io, "p", "r")
assert cells(io) == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[0,2],[["r",2]],null,1,1],[[1,0],[["t",4]],null,1,1],[[1,1],[["s",3]],null,1,1]]'
assert json.dumps([queue(io), [problems(L.OfShape, s) for s in (cpu, gpu, mmu, io)]], separators=(',', ':')) == '[[],[[],[],[],[]]]'

<a id="CNF-07"></a>
## CNF-07 · Removing an ignored spec moves nothing

In [ ]:
dsp = box("dsp", "abcd", 3)
assert json.dumps([beside(dsp, "a", "d", 180.0), beside(dsp, "a", "b", 270.0)], separators=(',', ':')) == '[true,false]'
assert cells(dsp) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1]]'
apart(dsp, "a", "b")
assert cells(dsp) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1]]'
assert json.dumps([queue(dsp), rules(dsp)], separators=(',', ':')) == '[[],[["a","d",180.0,false]]]'